In [1]:
%pip install torch torchvision scikit-learn pandas onnx onnxscript

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
import os
from torchvision.transforms import v2
from model import Model
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, log_loss

# 1. 하이퍼파라미터 및 환경 설정 (Hyperparameters and Environment Settings)


In [3]:
BATCH_SIZE = 32 # GPU 메모리에 따라 16, 32, 64 중 선택 (Choose from 16, 32, 64 depending on GPU memory)
EPOCHS = 50
LEARNING_RATE = 0.001
# CUDA(GPU) 사용 가능하면 CUDA, 아니면 MPS(Apple Silicon), 그마저도 아니면 CPU 사용 (Use CUDA if available, else MPS if available, else CPU)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")


# 2. 데이터 전처리 (Data Augmentation) (Data Preprocessing / Augmentation)

In [4]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)), # 이미지 크기 조정 (Resize images)
        transforms.RandomHorizontalFlip(), # 데이터 증강: 좌우 반전 (Data augmentation: Horizontal flip)
        transforms.ToTensor(), # 이미지를 텐서로 변환 (Convert image to tensor)
        # 정규화: ImageNet 평균과 표준편차 사용 (Normalize: Use ImageNet mean and std dev)
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'valid': transforms.Compose([
        transforms.Resize((224, 224)), # 이미지 크기 조정 (Resize images)
        transforms.ToTensor(), # 이미지를 텐서로 변환 (Convert image to tensor)
        # 정규화 (Normalize)
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

data_dir = './class-1' # 데이터셋 디렉토리 설정 (Set dataset directory)
# 데이터셋 불러오기 (Load datasets)
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x])
                  for x in ['train', 'valid']}
# 데이터 로더 설정 (Set up data loaders)
dataloaders = {x: DataLoader(image_datasets[x], batch_size=BATCH_SIZE, shuffle=True)
              for x in ['train', 'valid']}

# 3. 모델 설계 (Transfer Learning) (Model Design / Transfer Learning)


In [5]:
model=Model(len(image_datasets['train'].classes))

# 4. 손실 함수 (Loss Function)

In [10]:
criterion = nn.CrossEntropyLoss() # Cross-Entropy Loss 함수 정의 (Define Cross-Entropy Loss function)
optimizer=optim.Adam(model.head.parameters(),lr=LEARNING_RATE)

# 5. 학습 루프 (Training Loop)

In [11]:
print(f"학습 시작... (Device: {DEVICE})") # 학습 시작 메시지 출력 (Print training start message)
result={
    "train_accuracy_score":[],
    "train_loss":[],
    "train_precision_score":[],
    "train_recall_score":[],
    "train_f1_score":[],

    "valid_accuracy_score":[],
    "valid_loss":[],
    "valid_precision_score":[],
    "valid_recall_score":[],
    "valid_f1_score":[]
}
for epoch in range(EPOCHS): # 에포크 수만큼 반복 (Loop for the number of epochs)
    print(f'Epoch {epoch+1}/{EPOCHS}') # 현재 에포크 출력 (Print current epoch)
    print('-' * 10)

    for phase in ['train', 'valid']: # 'train' 및 'valid' 단계 반복 (Loop through 'train' and 'valid' phases)
        epoch_preds = [] # 현재 에포크 예측값 리스트 초기화 (Initialize list for current epoch's predictions)
        epoch_labels = [] # 현재 에포크 실제 레이블 리스트 초기화 (Initialize list for current epoch's true labels)
        running_loss = 0.0 # 현재 단계의 누적 손실 초기화 (Initialize running loss for current phase)
        running_corrects = 0 # 현재 단계의 누적 정답 수 초기화 (Initialize running correct count for current phase)

        if phase == 'train': # 학습 단계일 경우 모델을 학습 모드로 설정 (Set model to training mode for training phase)
            model.train()
        else: # 검증 단계일 경우 모델을 평가 모드로 설정 (Set model to evaluation mode for validation phase)
            model.eval()

        # 데이터 로더에서 배치 단위로 이미지와 레이블을 가져옴 (Get images and labels in batches from data loader)
        for inputs, labels in dataloaders[phase]:
            inputs = inputs.to(DEVICE) # 입력 데이터를 지정된 장치로 이동 (Move input data to specified device)
            labels = labels.to(DEVICE) # 레이블 데이터를 지정된 장치로 이동 (Move label data to specified device)

            optimizer.zero_grad() # 옵티마이저의 모든 그라디언트를 0으로 초기화 (Zero out gradients of the optimizer)

            # 학습 단계에서만 그라디언트 계산 활성화 (Enable gradient calculation only in training phase)
            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs) # 모델 예측 수행 (Perform model prediction)
                _, preds = torch.max(outputs, 1) # 가장 높은 확률을 가진 클래스 선택 (Select class with highest probability)
                loss = criterion(outputs, labels) # 손실 계산 (Calculate loss)

                if phase == 'train': # 학습 단계일 경우 (If in training phase)
                    loss.backward() # 역전파 수행 (Perform backpropagation)
                    optimizer.step() # 옵티마이저 파라미터 업데이트 (Update optimizer parameters)

            # 예측값과 실제 레이블 저장 (Store predictions and true labels)
            epoch_preds.extend(preds.cpu().numpy()) # 예측값을 CPU로 옮겨 numpy 배열로 변환 후 추가 (Move predictions to CPU, convert to numpy array, then extend)
            epoch_labels.extend(labels.cpu().numpy()) # 실제 레이블을 CPU로 옮겨 numpy 배열로 변환 후 추가 (Move true labels to CPU, convert to numpy array, then extend)
            running_loss += loss.item() * inputs.size(0) # 배치 손실을 누적 손실에 추가 (Add batch loss to running loss)
            running_corrects += torch.sum(preds == labels.data) # 올바른 예측 수 누적 (Accumulate correct predictions)

        # 에포크가 끝난 후 단계별 평균 손실 및 정확도 계산 (Calculate average loss and accuracy for the phase after epoch ends)
        epoch_loss = running_loss / len(image_datasets[phase])
        epoch_acc = accuracy_score(epoch_labels, epoch_preds)

        # 그 외 성능 지표 계산 (Calculate other performance metrics)
        phase_precision = precision_score(epoch_labels, epoch_preds, average='weighted', zero_division=0)
        phase_recall = recall_score(epoch_labels, epoch_preds, average='weighted', zero_division=0)
        phase_f1 = f1_score(epoch_labels, epoch_preds, average='weighted', zero_division=0)

        # 결과 저장 (Store results)
        result[f"{phase}_accuracy_score"].append(float(epoch_acc))
        result[f"{phase}_loss"].append(epoch_loss)
        result[f"{phase}_precision_score"].append(phase_precision)
        result[f"{phase}_recall_score"].append(phase_recall)
        result[f"{phase}_f1_score"].append(phase_f1)

        print(f'{phase.upper()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} Precision: {phase_precision:.4f} Recall: {phase_recall:.4f} F1-Score: {phase_f1:.4f}')

# 6. 모델 가중치 저장 (Save Model Weights)
torch.save(model.state_dict(), "model.pt") # 모델 가중치 저장 (Save model weights)
print("학습 완료 및 모델 저장 성공.") # 학습 완료 메시지 출력 (Print training completion message)



학습 시작... (Device: cpu)
Epoch 1/50
----------
TRAIN Loss: 1.1250 Acc: 0.2258 Precision: 0.2016 Recall: 0.2258 F1-Score: 0.2086
VALID Loss: 1.0908 Acc: 0.4444 Precision: 0.1975 Recall: 0.4444 F1-Score: 0.2735
Epoch 2/50
----------
TRAIN Loss: 0.9996 Acc: 0.5376 Precision: 0.5405 Recall: 0.5376 F1-Score: 0.5362
VALID Loss: 1.0180 Acc: 0.5556 Precision: 0.4444 Recall: 0.5556 F1-Score: 0.4444
Epoch 3/50
----------
TRAIN Loss: 0.9096 Acc: 0.7312 Precision: 0.7290 Recall: 0.7312 F1-Score: 0.7283
VALID Loss: 0.9534 Acc: 0.6667 Precision: 0.8095 Recall: 0.6667 F1-Score: 0.6380
Epoch 4/50
----------
TRAIN Loss: 0.8191 Acc: 0.8387 Precision: 0.8374 Recall: 0.8387 F1-Score: 0.8342
VALID Loss: 0.9339 Acc: 0.7778 Precision: 0.8519 Recall: 0.7778 F1-Score: 0.7704
Epoch 5/50
----------
TRAIN Loss: 0.7599 Acc: 0.8065 Precision: 0.8079 Recall: 0.8065 F1-Score: 0.8039
VALID Loss: 0.9299 Acc: 0.6667 Precision: 0.7111 Recall: 0.6667 F1-Score: 0.6667
Epoch 6/50
----------
TRAIN Loss: 0.6681 Acc: 0.8817 Prec

In [12]:

dummy_input = torch.randn(1, 3, 224, 224, device=DEVICE)
onnx_file_path = "model.onnx"
torch.onnx.export(
    model,                  # 이제 '가중치'가 아니라 '모델 객체'를 넘깁니다.
    dummy_input,            
    onnx_file_path,         
    export_params=True,     
    opset_version=12,       
    do_constant_folding=True,
    input_names=['input'],   
    output_names=['output']
)


In [13]:
import pandas as pd
df=pd.DataFrame(result)
df.index = df.index + 1
df.to_csv("result.csv",index_label="epoch")